In [2]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service as ChromeService
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import NoSuchElementException, WebDriverException

# CONFIGURATION
START_PAGE = 1          # change to resume from any page
END_PAGE = 2            # set to e.g. 2000000 for all, or a test value (2 for demo)
DELAY = 1.5             # seconds to wait for page load
OUTFILE = 'kohesio_projects.xlsx'

# SETUP SELENIUM
chrome_options = Options()
chrome_options.add_argument('--headless')   # Remove if you want to see browser
chrome_options.add_argument('--disable-gpu')
chrome_options.add_argument('--window-size=1920,1080')
driver = webdriver.Chrome(options=chrome_options)

base_url = "https://kohesio.ec.europa.eu/en/projects?page={}"

projects = []

def parse_project_blocks():
    data = {}
    # Find all blocks on right side (side content blocks)
    blocks = driver.find_elements(By.CSS_SELECTOR, '.kohesio-side-content-block')
    for block in blocks:
        try:
            title = block.find_element(By.CLASS_NAME, "title").text.strip().lower()
            # Each block has multiple content lines
            lines = block.find_elements(By.CLASS_NAME, "content")
            values = []
            for line in lines:
                values.append(line.text.strip())
            # Join multiple lines with " | "
            data[title] = " | ".join(values)
        except Exception:
            continue
    return data

def parse_project_card(card):
    # Get all summary fields on the listing page
    try:
        title_elem = card.find_element(By.CSS_SELECTOR, "h4 > a")
        title = title_elem.text.strip()
        url = title_elem.get_attribute("href")
    except:
        title, url = "", ""
    try:
        theme = card.find_element(By.CSS_SELECTOR, ".article--post__theme").text.strip()
    except:
        theme = ""
    try:
        budget = card.find_element(By.CSS_SELECTOR, ".article--post__amount .amount").text.strip()
    except:
        budget = ""
    try:
        years = card.find_element(By.CSS_SELECTOR, ".article--post__years .time-start-end").text.strip()
    except:
        years = ""
    try:
        teaser = card.find_element(By.CSS_SELECTOR, ".article--post__teaser").text.strip()
    except:
        teaser = ""
    try:
        country_flag = card.find_element(By.CSS_SELECTOR, ".flag").get_attribute("alt")
    except:
        country_flag = ""
    return {
        "title": title,
        "url": url,
        "theme": theme,
        "budget": budget,
        "years": years,
        "teaser": teaser,
        "country_flag": country_flag
    }

def save_projects_to_excel(projects, OUTFILE):
    df = pd.DataFrame(projects)
    df.to_excel(OUTFILE, index=False)

try:
    for page in range(START_PAGE, END_PAGE+1):
        print(f"Page {page}...")
        driver.get(base_url.format(page))
        time.sleep(DELAY + 1.5)  # Let JS load

        # Get all project cards
        cards = driver.find_elements(By.CSS_SELECTOR, ".article--post")
        print(f"  Found {len(cards)} projects on this page.")

        for idx, card in enumerate(cards):
            proj_data = parse_project_card(card)

            if not proj_data["url"]:
                continue

            print(f"    Scraping project {idx+1}/{len(cards)}: {proj_data['title']}")
            # Open detail in new tab (to not lose listing)
            driver.execute_script("window.open(arguments[0]);", proj_data["url"])
            driver.switch_to.window(driver.window_handles[-1])
            time.sleep(DELAY + 0.5)

            # Parse side blocks
            detail_data = parse_project_blocks()

            # Merge
            project_record = proj_data.copy()
            project_record.update(detail_data)
            projects.append(project_record)

            driver.close()
            driver.switch_to.window(driver.window_handles[0])

        # Save progress after each page
        save_projects_to_excel(projects, OUTFILE)
        print(f"  Progress saved ({len(projects)} projects)...")

except Exception as e:
    print(f"Exception occurred: {e}")

finally:
    driver.quit()
    print("Scraping finished.")
    save_projects_to_excel(projects, OUTFILE)


Page 1...
  Found 15 projects on this page.
    Scraping project 1/15: Regional project for the development of water and wastewater infrastructure in Cluj and Sălaj counties in 2014-2020
    Scraping project 2/15: M4 implementation of the section between Berettyóújfalu-Nagykereki (national border)
    Scraping project 3/15: “TEAM-UP: Progress in the quality of alternative childcare”
    Scraping project 4/15: Construction of the northern ring road of Krakow within S52
    Scraping project 5/15: Rehabilitation of the district heating system of Bucharest
    Scraping project 6/15: Preparation of the Sibiu – Pitesti Highway project and the construction of Sections 1, 4 and 5
    Scraping project 7/15: Water reservoir Racibórz Dolny on the river Oder in the Silesian voivodship (polder)
    Scraping project 8/15: Construction of the S7 expressway, p. Widoma – Kraków (c. Kraków Nowa Huta)
    Scraping project 9/15: Construction of the S-8 Wyszków Expressway – Białystok, pc. Wyszków – border 

In [3]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options

START_PAGE = 1
END_PAGE = 2
DELAY = 1.5
OUTFILE = 'kohesio_projects_fixed.xlsx'

chrome_options = Options()
chrome_options.add_argument('--headless')
chrome_options.add_argument('--disable-gpu')
chrome_options.add_argument('--window-size=1920,1080')
driver = webdriver.Chrome(options=chrome_options)

base_url = "https://kohesio.ec.europa.eu/en/projects?page={}"

projects = []

def parse_project_blocks():
    data = {}
    # Find all blocks on right side (side content blocks)
    blocks = driver.find_elements(By.CSS_SELECTOR, '.kohesio-side-content-block')
    for block in blocks:
        try:
            title = block.find_element(By.CLASS_NAME, "title").text.strip().lower()
            lines = block.find_elements(By.CLASS_NAME, "content")
            values = []
            for line in lines:
                # for "theme" and "intervention fields", anchor text is present
                anchors = line.find_elements(By.TAG_NAME, "a")
                if anchors:
                    anchor_texts = [a.text.strip() for a in anchors if a.text.strip()]
                    if anchor_texts:
                        values.extend(anchor_texts)
                        continue
                values.append(line.text.strip())
            data[title] = " | ".join(values)
        except Exception:
            continue
    return data

def parse_project_card(card):
    try:
        title_elem = card.find_element(By.CSS_SELECTOR, "h4 > a")
        title = title_elem.text.strip()
        url = title_elem.get_attribute("href")
    except:
        title, url = "", ""
    # These are only on card, but we'll prefer to pull from detail page if missing
    try:
        budget = card.find_element(By.CSS_SELECTOR, ".article--post__amount .amount").text.strip()
    except:
        budget = ""
    try:
        years = card.find_element(By.CSS_SELECTOR, ".article--post__years .time-start-end").text.strip()
    except:
        years = ""
    try:
        teaser = card.find_element(By.CSS_SELECTOR, ".article--post__teaser").text.strip()
    except:
        teaser = ""
    try:
        country_flag = card.find_element(By.CSS_SELECTOR, ".flag").get_attribute("alt")
    except:
        country_flag = ""
    return {
        "title": title,
        "url": url,
        "budget": budget,
        "years": years,
        "teaser": teaser,
        "country_flag": country_flag
    }

def save_projects_to_excel(projects, OUTFILE):
    df = pd.DataFrame(projects)
    df.to_excel(OUTFILE, index=False)

try:
    for page in range(START_PAGE, END_PAGE+1):
        print(f"Page {page}...")
        driver.get(base_url.format(page))
        time.sleep(DELAY + 1.5)  # Let JS load

        cards = driver.find_elements(By.CSS_SELECTOR, ".article--post")
        print(f"  Found {len(cards)} projects on this page.")

        for idx, card in enumerate(cards):
            proj_data = parse_project_card(card)
            if not proj_data["url"]:
                continue
            print(f"    Scraping project {idx+1}/{len(cards)}: {proj_data['title']}")
            driver.execute_script("window.open(arguments[0]);", proj_data["url"])
            driver.switch_to.window(driver.window_handles[-1])
            time.sleep(DELAY + 0.5)
            detail_data = parse_project_blocks()

            # Special: always extract theme from detail page if available!
            theme = detail_data.get("theme", "")
            intervention_fields = detail_data.get("intervention fields", "")
            # Add these explicitly to top-level columns
            proj_data["theme"] = theme
            proj_data["intervention_fields"] = intervention_fields

            # Merge all detail_data (blocks)
            project_record = proj_data.copy()
            project_record.update(detail_data)
            projects.append(project_record)

            driver.close()
            driver.switch_to.window(driver.window_handles[0])

        save_projects_to_excel(projects, OUTFILE)
        print(f"  Progress saved ({len(projects)} projects)...")

except Exception as e:
    print(f"Exception occurred: {e}")

finally:
    driver.quit()
    print("Scraping finished.")
    save_projects_to_excel(projects, OUTFILE)


Page 1...
Exception occurred: HTTPConnectionPool(host='localhost', port=50079): Read timed out. (read timeout=120)
Scraping finished.


In [1]:
import requests
import pandas as pd

all_projects = []
base_url = "https://kohesio.ec.europa.eu/api/projects?page={page}"

for page in range(1, 101):  # Adjust range as needed
    url = base_url.format(page=page)
    resp = requests.get(url)
    if resp.status_code != 200 or not resp.json().get("results"):
        break  # No more results
    data = resp.json()["results"]
    all_projects.extend(data)

# Save to Excel or CSV
df = pd.DataFrame(all_projects)
df.to_csv("kohesio_projects.csv", index=False)


In [2]:
import requests
import pandas as pd
import time

all_projects = []
base_url = "https://kohesio.ec.europa.eu/api/projects?page={}"  # Update as needed
max_pages = 15  # Try high number; script will auto-stop

for page in range(1, max_pages + 1):
    url = base_url.format(page)
    r = requests.get(url)
    if r.status_code != 200:
        print(f"Stopped at page {page} (status: {r.status_code})")
        break
    data = r.json()
    results = data.get('results') or data  # Adjust if API returns plain list
    if not results or len(results) == 0:
        print(f"No more results at page {page}")
        break
    all_projects.extend(results)
    print(f"Fetched page {page}: {len(results)} projects (total: {len(all_projects)})")
    time.sleep(0.2)  # Be polite

# Save as CSV/Excel
df = pd.DataFrame(all_projects)
df.to_csv('kohesio_projects_all.csv', index=False)
df.to_excel('kohesio_projects_all.xlsx', index=False)
print("Saved!")


Fetched page 1: 3 projects (total: 3)
Fetched page 2: 3 projects (total: 6)
Fetched page 3: 3 projects (total: 9)
Fetched page 4: 3 projects (total: 12)
Fetched page 5: 3 projects (total: 15)
Fetched page 6: 3 projects (total: 18)
Fetched page 7: 3 projects (total: 21)
Fetched page 8: 3 projects (total: 24)
Fetched page 9: 3 projects (total: 27)
Fetched page 10: 3 projects (total: 30)
Fetched page 11: 3 projects (total: 33)
Fetched page 12: 3 projects (total: 36)
Fetched page 13: 3 projects (total: 39)
Fetched page 14: 3 projects (total: 42)
Fetched page 15: 3 projects (total: 45)
Saved!


In [3]:
import requests
import pandas as pd
import time

all_projects = []
base_url = "https://kohesio.ec.europa.eu/api/projects?page={}"  # Or your real API

for page in range(1, 2000):  # Test on first 4 pages!
    url = base_url.format(page)
    resp = requests.get(url)
    data = resp.json()
    # Adjust according to your actual API response
    projects = data.get('list') or data.get('results') or data
    if not projects or not isinstance(projects, list):
        print("No more projects or wrong key:", projects)
        break
    all_projects.extend(projects)
    print(f"Page {page}: {len(projects)} projects")
    time.sleep(0.2)

# Save real projects data
df = pd.DataFrame(all_projects)
df.to_csv('projects_proper.csv', index=False)
df.to_excel('projects_proper.xlsx', index=False)
print("Saved!")


Page 1: 1000 projects
Page 2: 1000 projects
Page 3: 1000 projects
Page 4: 1000 projects
Saved!


In [1]:
import requests
import pandas as pd
import time

all_projects = []
base_url = "https://kohesio.ec.europa.eu/api/projects?page={}"

for page in range(1, 5):  # Start with 4 pages for testing!
    url = base_url.format(page)
    resp = requests.get(url)
    data = resp.json()
    # The API key for list of projects is likely 'list' or 'results'
    projects = data.get('list') or data.get('results') or data
    if not projects or not isinstance(projects, list):
        print("No more projects or wrong key:", projects)
        break

    # Print keys of the first project for debugging
    if projects and len(all_projects) == 0:
        print("Sample project keys:", projects[0].keys())

    # Extract relevant fields
    for proj in projects:
        project_info = {
            "title": proj.get("title") or proj.get("name"),
            "country": proj.get("country") or proj.get("countryCode"),
            "sector": proj.get("mainTheme") or proj.get("thematicObjective") or proj.get("sector"),
            "start_date": proj.get("startDate"),
            "end_date": proj.get("endDate"),
            "total_budget": proj.get("totalBudget"),
            "eu_contribution": proj.get("euCoFinancing"),
            "programme": proj.get("programme"),
            "beneficiary": proj.get("beneficiary"),
            "project_id": proj.get("projectId") or proj.get("id") or proj.get("code"),
        }
        all_projects.append(project_info)
    print(f"Page {page}: {len(projects)} projects")
    time.sleep(0.2)

# Save to file
df = pd.DataFrame(all_projects)
df.to_csv('projects_proper1csv', index=False)
df.to_excel('projects_proper1.xlsx', index=False)
print("Saved!")


Sample project keys: dict_keys(['link', 'item', 'snippet', 'labels', 'originalLabels', 'descriptions', 'orignalDescriptions', 'startTimes', 'endTimes', 'euBudgets', 'totalBudgets', 'images', 'copyrightImages', 'coordinates', 'objectiveIds', 'countrycode'])
Page 1: 1000 projects
Page 2: 1000 projects
Page 3: 1000 projects
Page 4: 1000 projects
Saved!


In [3]:
import requests
import pandas as pd
import time

all_projects = []
base_url = "https://kohesio.ec.europa.eu/api/projects?page={}"  

for page in range(1, 2000):  
    url = base_url.format(page)
    resp = requests.get(url)
    data = resp.json()
    # Adjust according to your actual API response
    projects = data.get('list') or data.get('results') or data
    if not projects or not isinstance(projects, list):
        print("No more projects or wrong key:", projects)
        break
    all_projects.extend(projects)
    print(f"Page {page}: {len(projects)} projects")
    time.sleep(0.2)

# Save real projects data
df = pd.DataFrame(all_projects)
df.to_csv('European Union Cohesion Fund.csv', index=False)
df.to_excel('European Union Cohesion Fund.xlsx', index=False)
print("Saved!")

Page 1: 1000 projects
Page 2: 1000 projects
Page 3: 1000 projects
Page 4: 1000 projects
Page 5: 1000 projects
Page 6: 1000 projects
Page 7: 1000 projects
Page 8: 1000 projects
Page 9: 1000 projects
Page 10: 1000 projects
Page 11: 1000 projects
Page 12: 1000 projects
Page 13: 1000 projects
Page 14: 1000 projects
Page 15: 1000 projects
Page 16: 1000 projects
Page 17: 1000 projects
Page 18: 1000 projects
Page 19: 1000 projects
Page 20: 1000 projects
Page 21: 1000 projects
Page 22: 1000 projects
Page 23: 1000 projects
Page 24: 1000 projects
Page 25: 1000 projects
Page 26: 1000 projects
Page 27: 1000 projects
Page 28: 1000 projects
Page 29: 1000 projects
Page 30: 1000 projects
Page 31: 1000 projects
Page 32: 1000 projects
Page 33: 1000 projects
Page 34: 1000 projects
Page 35: 1000 projects
Page 36: 1000 projects
Page 37: 1000 projects
Page 38: 1000 projects
Page 39: 1000 projects
Page 40: 1000 projects
Page 41: 1000 projects
Page 42: 1000 projects
Page 43: 1000 projects
Page 44: 1000 projec

ConnectTimeout: HTTPSConnectionPool(host='kohesio.ec.europa.eu', port=443): Max retries exceeded with url: /api/projects?page=69 (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x000001F08DDD16A0>, 'Connection to kohesio.ec.europa.eu timed out. (connect timeout=None)'))

In [1]:
import requests
import pandas as pd
import time

# Continue scraping from where you left off
start_page = 1809
end_page = 2000  # or whatever final page you expect

# Load the existing data if needed
try:
    existing_df = pd.read_csv('European Union Cohesion Fund.csv')
    all_projects = existing_df.to_dict(orient='records')
    print(f"Loaded {len(existing_df)} existing records.")
except FileNotFoundError:
    all_projects = []
    print("No existing file found. Starting fresh.")

base_url = "https://kohesio.ec.europa.eu/api/projects?page={}"

for page in range(start_page, end_page):
    url = base_url.format(page)
    try:
        resp = requests.get(url, timeout=10)
        data = resp.json()
    except Exception as e:
        print(f"Error fetching page {page}: {e}")
        break

    # Handle different possible response formats
    projects = data.get('list') or data.get('results') or data
    if not projects or not isinstance(projects, list):
        print("No more projects or unexpected data structure:", projects)
        break

    all_projects.extend(projects)
    print(f"Page {page}: {len(projects)} projects")
    time.sleep(0.2)  # be respectful to server

# Save the full combined dataset
df = pd.DataFrame(all_projects)
df.to_csv('European Union Cohesion Fund.csv', index=False)
df.to_excel('European Union Cohesion Fund.xlsx', index=False)
print("Saved!")


No existing file found. Starting fresh.
Page 1809: 1000 projects
Error fetching page 1810: HTTPSConnectionPool(host='kohesio.ec.europa.eu', port=443): Max retries exceeded with url: /api/projects?page=1810 (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x000001F0E68F3E90>, 'Connection to kohesio.ec.europa.eu timed out. (connect timeout=10)'))
Saved!
